# Lab | Music recommendations

- First re-run everything in this notebook to ensure you're comfortable with the concepts of similar audio recommendation systems based on RAG.
- Using music datasets from [this](https://github.com/Yuan-ManX/ai-audio-datasets?tab=readme-ov-file#m) github repo, create a local RAG to recommend sons based on users preferences. Example dataset from that link could be [this](https://zenodo.org/records/5794629) Artificial multitrack audio data. Feel free to find you're own datasets online, or combine the dataset used in this lab with a few you found to make some recommendations.
- Go ahead and build something great in 4 hours.

This lab demonstrate how to use Pinecone as the vector DB within an audio search application. Audio search can be used to find songs and metadata within a catalog, finding similar sounds in an audio library, or detecting who's speaking in an audio file.

We will index a set of audio recordings as vector embeddings. These vector embeddings are rich, mathematical representations of the audio recordings, making it possible to determine how similar the recordings are to one another. We will then take some new (unseen) audio recording, search through the index to find the most similar matches, and play the returned audio in this notebook.

# Install Dependencies

In [45]:
!pip install librosa
!pip install panns-inference

In [46]:
 !pip install -qU pinecone-client==3.1.0 panns-inference datasets librosa

# Load Dataset

In this demo, we will use audio from the *ESC-50 dataset* — a labeled collection of 2000 environmental audio recordings, which are 5-second-long each. The dataset can be loaded from the HuggingFace model hub as follows:

In [3]:
from datasets import load_dataset

# load the dataset from huggingface model hub
data = load_dataset("ashraq/esc50", split="train")
data

Repo card metadata block was not found. Setting CardData to empty.


Dataset({
    features: ['filename', 'fold', 'target', 'category', 'esc10', 'src_file', 'take', 'audio'],
    num_rows: 2000
})

The audios in the dataset are sampled at 44100Hz and loaded into NumPy arrays. Let's take a look.

In [4]:
# select the audio feature and display top three
audios = data["audio"]
audios[:3]

[{'path': None,
  'array': array([0., 0., 0., ..., 0., 0., 0.]),
  'sampling_rate': 44100},
 {'path': None,
  'array': array([-0.01184082, -0.10336304, -0.14141846, ...,  0.06985474,
          0.04049683,  0.00274658]),
  'sampling_rate': 44100},
 {'path': None,
  'array': array([-0.00695801, -0.01251221, -0.01126099, ...,  0.215271  ,
         -0.00875854, -0.28903198]),
  'sampling_rate': 44100}]

We only need the Numpy arrays as these contain all of the audio data. We will later input these Numpy arrays directly into our embedding model to generate audio embeddings.

In [10]:
import numpy as np

# select only the audio data from the dataset and store in a numpy array
audios = np.array([a["array"] for a in data["audio"]])

# Load Audio Embedding Model

We will use an audio tagging model trained from *PANNs: Large-Scale Pretrained Audio Neural Networks for Audio Pattern Recognition* paper to generate our audio embeddings. We use the *panns_inference* Python package, which provides an easy interface to load and use the model.

In [13]:
import os
import requests

# Define the directory and file path based on your specified path
directory = '/Users/marinacastilloariza/Desktop/AI_Work/8_Week/lab-music-recommendations/data'
file_path = f"{directory}/class_labels_indices.csv"

# Create the directory if it doesn't exist
os.makedirs(directory, exist_ok=True)

# Download the file if it’s not already there
if not os.path.exists(file_path):
    url = "http://storage.googleapis.com/us_audioset/youtube_corpus/v1/csv/class_labels_indices.csv"
    response = requests.get(url)
    with open(file_path, 'wb') as f:
        f.write(response.content)
    print("File downloaded successfully.")
else:
    print("File already downloaded.")


File downloaded successfully.


## Initializing the Index

Now we need a place to store these embeddings and enable a efficient vector search through them all. To do that we use Pinecone, we can get a [free API key](https://app.pinecone.io/) and enter it below where we will initialize our connection to Pinecone and create a new index.

In [16]:
from dotenv import load_dotenv, find_dotenv
import os
_ = load_dotenv(find_dotenv())

# Prompt for API keys
PINECONE_API_KEY = input("Please enter your Pinecone API key: ")
OPENAI_API_KEY = input("Please enter your OpenAI API key: ")

# Initialize the Pinecone client
pc = Pinecone(api_key=PINECONE_API_KEY)




In [17]:
import os
from pinecone import Pinecone

# configure client
pc = Pinecone(api_key=PINECONE_API_KEY)

# Define the index name and configuration
index_name = "audio-search-demo"
cloud = 'aws'  # Default cloud provider
region = 'us-east-1'  # Default region

Now we setup our index specification, this allows us to define the cloud provider and region where we want to deploy our index. You can find a list of all [available providers and regions here](https://docs.pinecone.io/docs/projects).

In [18]:
from pinecone import ServerlessSpec

cloud = os.environ.get('PINECONE_CLOUD') or 'aws'
region = os.environ.get('PINECONE_REGION') or 'us-east-1'

spec = ServerlessSpec(cloud=cloud, region=region)

Create the index:

In [19]:
index_name = "audio-search-demo"

In [21]:
import time
from pinecone import ServerlessSpec

# Define the index specification
spec = ServerlessSpec(cloud='aws', region='us-east-1')

try:
    # Try creating the index directly
    pc.create_index(index_name, dimension=2048, metric='cosine', spec=spec)
    # Wait for the index to initialize
    while not pc.describe_index(index_name).status['ready']:
        time.sleep(1)
    print("Index created successfully.")
except Exception as e:
    print("Index might already exist or there was an issue:", e)

# Connect to the index
index = pc.Index(index_name)

# View index stats to confirm connection
print(index.describe_index_stats())


Index created successfully.
{'dimension': 2048,
 'index_fullness': 0.0,
 'namespaces': {},
 'total_vector_count': 0}


In [24]:
import os
import requests

# Define the correct directory and file path
directory = '/Users/marinacastilloariza/panns_data'
file_path = f"{directory}/class_labels_indices.csv"

# Create the directory if it doesn't exist
os.makedirs(directory, exist_ok=True)

# Download the file if it’s not already there
if not os.path.exists(file_path):
    url = "http://storage.googleapis.com/us_audioset/youtube_corpus/v1/csv/class_labels_indices.csv"
    response = requests.get(url)
    with open(file_path, 'wb') as f:
        f.write(response.content)
    print("File downloaded successfully.")
else:
    print("File already exists.")


File downloaded successfully.


In [26]:
# Define the path for the checkpoint file
checkpoint_path = '/Users/marinacastilloariza/panns_data/Cnn14_mAP=0.431.pth'

# Download the checkpoint file if it’s not already there
if not os.path.exists(checkpoint_path):
    url = "https://zenodo.org/record/3987831/files/Cnn14_mAP=0.431.pth?download=1"
    response = requests.get(url)
    with open(checkpoint_path, 'wb') as f:
        f.write(response.content)
    print("Checkpoint file downloaded successfully.")
else:
    print("Checkpoint file already exists.")


Checkpoint file downloaded successfully.


In [27]:
from panns_inference import AudioTagging

# Load the model with the correct checkpoint path
model = AudioTagging(checkpoint_path=checkpoint_path, device='cpu')


Checkpoint path: /Users/marinacastilloariza/panns_data/Cnn14_mAP=0.431.pth
Using CPU.


In [28]:
from panns_inference import AudioTagging

# Load the audio embedding model with CPU configuration
model = AudioTagging(checkpoint_path=None, device='cpu')


Checkpoint path: /Users/marinacastilloariza/panns_data/Cnn14_mAP=0.431.pth
Using CPU.


# Generate Embeddings and Upsert

Now we generate the embeddings using the audio embedding model. We must do this in batches as processing all items at once will exhaust machine memory limits and API request limits.

In [29]:
from tqdm.auto import tqdm

# we will use batches of 64
batch_size = 64

for i in tqdm(range(0, len(audios), batch_size)):
    # find end of batch
    i_end = min(i+batch_size, len(audios))
    # extract batch
    batch = audios[i:i_end]
    # generate embeddings for all the audios in the batch
    _, emb = model.inference(batch)
    # create unique IDs
    ids = [f"{idx}" for idx in range(i, i_end)]
    # add all to upsert list
    to_upsert = list(zip(ids, emb.tolist()))
    # upsert/insert these records to pinecone
    _ = index.upsert(vectors=to_upsert)

# check that we have all vectors in index
index.describe_index_stats()

  0%|          | 0/32 [00:00<?, ?it/s]

{'dimension': 2048,
 'index_fullness': 0.0,
 'namespaces': {'': {'vector_count': 2000}},
 'total_vector_count': 2000}

We now have *2000* audio records indexed in Pinecone, we're ready to begin querying.

# Querying

Let's first listen to an audio from our dataset. We will generate embeddings for the audio and use it to find similar audios from the Pinecone index.

In [30]:
from IPython.display import Audio, display

# we set an audio number to select from the dataset
audio_num = 400
# get the audio data of the audio number
query_audio = data[audio_num]["audio"]["array"]
# get the category of the audio number
category = data[audio_num]["category"]
# print the category and play the audio
print("Query Audio:", category)
Audio(query_audio, rate=44100)

Query Audio: car_horn


We have got the sound of a car horn. Let's generate an embedding for this sound.

In [31]:
# reshape query audio
query_audio = query_audio[None, :]
# get the embeddings for the audio from the model
_, xq = model.inference(query_audio)
xq.shape

(1, 2048)

We have now converted the audio into a 2048-dimension vector the same way we did for all the other audio we indexed. Let's use this to query our Pinecone index.

In [32]:
# query pinecone index with the query audio embeddings
results = index.query(vector=xq.tolist(), top_k=3)
results

{'matches': [{'id': '400', 'score': 0.999442637, 'values': []},
             {'id': '1667', 'score': 0.842259, 'values': []},
             {'id': '1666', 'score': 0.831919074, 'values': []}],
 'namespace': '',
 'usage': {'read_units': 5}}

Notice that the top result is the audio number 400 from our dataset, which is our query audio (the most similar item should always be the query itself). Let's listen to the top three results. 

In [33]:
# play the top 3 similar audios
for r in results["matches"]:
    # select the audio data from the databse using the id as an index
    a = data[int(r["id"])]["audio"]["array"]
    display(Audio(a, rate=44100))

We have great results, everything aligns with what seems to be a busy city street with car horns.

Let's write a helper function to run the queries using audio from our dataset easily. We do not need to embed these audio samples again as we have already, they are just stored in Pinecone. So, we specify the `id` of the query audio to search with and tell Pinecone to search with that.

In [34]:
def find_similar_audios(id):
    print("Query Audio:")
    # select the audio data from the databse using the id as an index
    query_audio = data[id]["audio"]["array"]
    # play the query audio
    display(Audio(query_audio, rate=44100))
    # query pinecone index with the query audio id
    result = index.query(id=str(id), top_k=5)
    print("Result:")
    # play the top 5 similar audios
    for r in result["matches"]:
        a = data[int(r["id"])]["audio"]["array"]
        display(Audio(a, rate=44100))

In [35]:
find_similar_audios(1642)

Query Audio:


Result:


Here we return a set of revving motors (they seem to either be vehicles or lawnmowers).

In [36]:
find_similar_audios(452)

Query Audio:


Result:


And now a more relaxing set of birds chirping in nature.

Let's use another audio sample from elsewhere (eg not this dataset) and see how the search performs with this.

In [37]:
!wget https://storage.googleapis.com/audioset/miaow_16k.wav

zsh:1: command not found: wget


We can load the audio into a Numpy array as follows:

In [42]:
import librosa

# Load the external audio file
audio_path = '/Users/marinacastilloariza/Desktop/AI_Work/8_Week/lab-music-recommendations/data/miaow_16k.wav'
external_audio, _ = librosa.load(audio_path, sr=44100)

# Display the external audio
print("External Query Audio")
display(Audio(external_audio, rate=44100))

# Reshape and create embeddings for the external audio
external_audio = external_audio[None, :]
_, xq = model.inference(external_audio)

# Query the Pinecone index with the external audio embedding
results = index.query(vector=xq.tolist(), top_k=3)

# Display the results
print("Top 3 Similar Audios in Index:")
for r in results["matches"]:
    similar_audio = data[int(r["id"])]["audio"]["array"]
    display(Audio(similar_audio, rate=44100))


External Query Audio


Top 3 Similar Audios in Index:


Now we generate the embeddings for this audio and query the Pinecone index.

In [43]:
# reshape query audio
query_audio = a[None, :]
# get the embeddings for the audio from the model
_, xq = model.inference(query_audio)

# query pinecone index with the query audio embeddings
results = index.query(vector=xq.tolist(), top_k=3)

# play the top 3 similar audios
for r in results["matches"]:
    a = data[int(r["id"])]["audio"]["array"]
    display(Audio(a, rate=44100))

Our audio search application has identified a set of similar cat sounds, which is excellent.

# Delete the Index

Delete the index once you are sure that you do not want to use it anymore. Once the index is deleted, you cannot use it again.

In [44]:
pc.delete_index(index_name)

In [2]:
import numpy as np
from datasets import load_dataset

# Load the audio dataset from Hugging Face
data = load_dataset("ashraq/esc50", split="train")

# Select only the audio data from the dataset and store it in a NumPy array
audios = np.array([a["array"] for a in data["audio"]])

# Set up the path for model checkpoint
import os
import requests

# Download checkpoint if not present
directory = '/Users/marinacastilloariza/panns_data'
os.makedirs(directory, exist_ok=True)
checkpoint_path = f"{directory}/Cnn14_mAP=0.431.pth"

if not os.path.exists(checkpoint_path):
    url = "https://zenodo.org/record/3987831/files/Cnn14_mAP=0.431.pth?download=1"
    response = requests.get(url)
    with open(checkpoint_path, 'wb') as f:
        f.write(response.content)
    print("Checkpoint file downloaded successfully.")
else:
    print("Checkpoint file already exists.")

# Load the audio embedding model
from panns_inference import AudioTagging
model = AudioTagging(checkpoint_path=checkpoint_path, device='cpu')

# Set up Pinecone client and initialize index
from pinecone import Pinecone, ServerlessSpec

# Prompt for Pinecone API key
PINECONE_API_KEY = input("Please enter your Pinecone API key: ")

# Initialize Pinecone client
pc = Pinecone(api_key=PINECONE_API_KEY)

# Define and create the Pinecone index
index_name = "audio-search-demo"
spec = ServerlessSpec(cloud='aws', region='us-east-1')
try:
    pc.create_index(index_name, dimension=2048, metric='cosine', spec=spec)
    print("Index created successfully.")
except Exception as e:
    print("Index might already exist or there was an issue:", e)

# Connect to the index
index = pc.Index(index_name)

# Generate and upsert audio embeddings with metadata
from tqdm.auto import tqdm

batch_size = 64
for i in tqdm(range(0, len(audios), batch_size)):
    i_end = min(i + batch_size, len(audios))
    batch = audios[i:i_end]
    _, emb = model.inference(batch)
    to_upsert = [(f"{idx}", vector.tolist(), {"category": data[idx]["category"]}) for idx, vector in zip(range(i, i_end), emb)]
    _ = index.upsert(vectors=to_upsert)

# Check that all vectors were upserted
print(index.describe_index_stats())

# Load and query new audio
import librosa
from IPython.display import Audio, display

# Load the new audio file
audio_path = '/Users/marinacastilloariza/Desktop/AI_Work/8_Week/lab-music-recommendations/data/ocean.wav'
new_audio, _ = librosa.load(audio_path, sr=44100)

# Display the new audio
print("New Query Audio - Ocean Sound")
display(Audio(new_audio, rate=44100))

# Generate embedding and query Pinecone
new_audio = new_audio[None, :]
_, xq = model.inference(new_audio)
results = index.query(vector=xq.tolist(), top_k=3)

# Display the top similar audio results
print("Top 3 Similar Audio Matches:")
for r in results["matches"]:
    similar_audio = data[int(r["id"])]["audio"]["array"]
    display(Audio(similar_audio, rate=44100))


Repo card metadata block was not found. Setting CardData to empty.


Checkpoint file already exists.
Checkpoint path: /Users/marinacastilloariza/panns_data/Cnn14_mAP=0.431.pth


/opt/anaconda3/lib/python3.12/site-packages/panns_inference/inference.py:55: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path, map_locat

Using CPU.
Index created successfully.


  0%|          | 0/32 [00:00<?, ?it/s]

{'dimension': 2048,
 'index_fullness': 0.0,
 'namespaces': {'': {'vector_count': 2000}},
 'total_vector_count': 2000}
New Query Audio - Ocean Sound


Top 3 Similar Audio Matches:
